# Bursting with the high-pass selector

Companion code for *Types of Bursting with a Two-Block Spiking Primitive*,
Section IV-B. **This notebook produces Fig. 3.**

The output equation $y = \mathrm{sig}_k(u + y - x)$ is implicit and multi-valued for
$k > 1$. Rather than regularizing it with a fast low-pass, which is stiff, we integrate the
exact hybrid form. With the saturation sigmoid the two outer branches are $y = +1$ and
$y = -1$, valid respectively while

$$x \le u + 1 - \tfrac{1}{k} \qquad \text{and} \qquad x \ge u - 1 + \tfrac{1}{k} .$$

The two windows overlap for $k > 1$, and that overlap is the hysteresis which makes the
relaxation cycle. Branch switching is handled by a `ContinuousCallback`, so the vector
field is linear everywhere and `Tsit5` is enough.

Two things are checked, in the order of the paper: the exogenous drive together with the
quasi-static spike count, and the cascade of two copies of the primitive.

In [ ]:
using Plots, LaTeXStrings, DifferentialEquations, DiffEqCallbacks
using Printf, Plots.PlotMeasures

gr(guidefontsize = 14, tickfontsize = 12, legendfontsize = 12, margin = 5Plots.mm, grid = true)
myBlue   = RGBA(131/255, 174/255, 218/255, 1)
myPurple = RGBA(169/255,  90/255, 179/255, 1)
myRed    = RGBA(158/255,   3/255,   8/255, 1)
myGray   = RGBA(150/255, 150/255, 150/255, 1)
default(fmt = :png);

## One loop, integrated exactly

In [ ]:
Base.@kwdef mutable struct HPLoop
    k::Float64   = 3.0        # sigmoid gain, k > 1 is needed to fire at all
    tau::Float64 = 1.0        # high-pass time constant
    u::Function  = t -> 0.0   # external input, at the sigmoid port
    y::Float64   = 1.0        # discrete branch, +1 or -1
end

hp_rhs!(dx, x, p::HPLoop, t) = (dx[1] = (-x[1] + p.y) / p.tau; nothing)

# the running branch stays valid until its saturation window is left
function hp_cond(x, t, integ)
    p  = integ.p
    ue = p.u(t)
    p.y > 0 ? x[1] - (ue + 1 - 1/p.k) : (ue - 1 + 1/p.k) - x[1]
end
hp_flip!(integ) = (integ.p.y = -integ.p.y; nothing)

"""
    simulate(p::HPLoop; tspan, x0, y0)

Integrate one loop. Returns the solution together with the saved branch history
`(ts, ys)`, since `y` is a discrete state and does not live in `sol`.
"""
function simulate(p::HPLoop; tspan = (0.0, 300.0), x0 = 0.0, y0 = 1.0, dtmax = 0.05)
    p.y = y0
    sv  = SavedValues(Float64, Float64)
    cb  = CallbackSet(ContinuousCallback(hp_cond, hp_flip!),
                      SavingCallback((x, t, integ) -> integ.p.y, sv))
    sol = solve(ODEProblem(hp_rhs!, [x0], tspan, p), Tsit5();
                callback = cb, abstol = 1e-10, reltol = 1e-9, dtmax = dtmax)
    return sol, sv.t, sv.saveval
end

"""Spike time: each downward switch of the branch, that is the end of a +1 plateau."""
spiketimes(ts, ys) = [ts[i] for i in 1:length(ts)-1 if ys[i] > 0 && ys[i+1] < 0]

"""Every node of the loop along a solution."""
function signals(sol, ts, ys, p::HPLoop)
    x = [sol(t)[1] for t in ts]
    return (t = ts, x = x, y = ys, w = ys .- x, u = p.u.(ts))
end

In [ ]:
# ---------------------------------------------------------------------
#  Closed forms for the saturation sigmoid, used in the checks below
# ---------------------------------------------------------------------

tplus(u, k, tau)  = tau * log((2k - 1 - k*abs(u)) / (1 - k*abs(u)))
tminus(u, k, tau) = tau * log((2k - 1 + k*abs(u)) / (1 + k*abs(u)))

"""Tonic rate inside the firing band, f = 1/(t_+ + t_-). Zero outside."""
fclosed(u; k = 3.0, tau = 1.0) =
    k*abs(u) >= 1 ? 0.0 : 1 / (tplus(u, k, tau) + tminus(u, k, tau))

rheobase_sat(k) = 1 / k       # band edge for the saturation sigmoid

@printf("k = 3:  u_th = %.4f,  f_max = %.5f\n", rheobase_sat(3.0), fclosed(0.0))

## Exogenous drive gives parabolic bursting

Drive the loop with a slow input of period much longer than $\tau$. Inside the band the
cell fires, outside it rests, and the rate vanishes at both edges. Nothing is added to the
model. The spike count per burst follows the quasi-static prediction
$N = \int_\mathrm{burst} f(u(t))\,dt$, which the first cell checks against the counted
spikes.

A symmetric drive gives two bursts per slow period, one on the way up and one on the way
down. One burst per period requires entering and leaving the band from the same side.

In [ ]:
# --- symmetric slow sine: two bursts per slow period
k, tau = 3.0, 1.0
A, fs  = 0.8, 0.004
p           = HPLoop(k = k, tau = tau, u = t -> A * sin(2pi * fs * t))
sol, ts, ys = simulate(p; tspan = (0.0, 2/fs), x0 = 0.0, dtmax = 0.02)
s           = signals(sol, ts, ys, p)
sp          = spiketimes(ts, ys)

tt    = range(0, 1/fs; length = 200_000)
Npred = sum(fclosed(A*sin(2pi*fs*t); k = k, tau = tau) for t in tt) * step(tt)
@printf("spikes per slow period:  quasi-static %.2f,  counted %.1f\n", Npred, length(sp)/2)

p1 = plot(s.t, s.u; lw = 2, c = :black, ylabel = L"u", legend = false)
hline!(p1, [-1/k, 1/k]; ls = :dash, c = myRed, label = false)
p2 = plot(s.t, s.w; lw = 1.5, c = myBlue,   ylabel = L"w", legend = false)
p3 = plot(s.t, s.y; lw = 1.5, c = myPurple, ylabel = L"y", xlabel = L"t/\tau",
          ylims = (-1.1, 1.1), legend = false)
plot(p1, p2, p3; layout = (3, 1), size = (950, 620), link = :x,
     title = ["symmetric drive, A = $A, f_slow = $fs" "" ""], titlefontsize = 11)

In [ ]:
# --- biased sine: exactly one burst per slow period
u0, A2 = 0.5, 0.7
@printf("u0 - A = %.3f inside the band: %s,   u0 + A = %.3f outside: %s\n",
        u0 - A2, abs(u0 - A2) < 1/k, u0 + A2, u0 + A2 > 1/k)

p = HPLoop(k = k, tau = tau, u = t -> u0 + A2 * sin(2pi * fs * t))
sol, ts, ys = simulate(p; tspan = (0.0, 2/fs), x0 = 1.0, dtmax = 0.02)
s = signals(sol, ts, ys, p)

p1 = plot(s.t, s.u; lw = 2, c = :black, ylabel = L"u", legend = false)
hline!(p1, [-1/k, 1/k]; ls = :dash, c = myRed, label = false)
p2 = plot(s.t, s.y; lw = 1.5, c = myPurple, ylabel = L"y", xlabel = L"t/\tau",
          ylims = (-1.1, 1.1), legend = false)
plot(p1, p2; layout = (2, 1), size = (950, 440), link = :x,
     title = ["biased drive, one burst per slow period" ""], titlefontsize = 11)

## The cascade: two copies of the primitive

The alternative is to replace the external drive by a second copy of the same two blocks,
running at $\tau_s \gg \tau$ and biased inside its own firing band so that it fires slowly.
Two dynamic states in total, and no new block type.

The switch selects which signal of the slow loop drives the fast one, and the position of
that switch selects the burst class: the filter state $x_s$ gives a smooth ramp and a
parabolic burst, the feedback signal $w_s$ gives a decaying plateau and an accelerating
burst, and the output $y_s$ gives a square wave, an abrupt onset and offset and a flat
intra-burst rate. The last one requires the bias $u_0$, since exactly one of the two
plateaus has to sit inside the band.

The return gain $\alpha$ closes the path from the fast loop back to the slow one. It is zero over the first half of Fig. 3 and steps to 0.2 over the second half.

In [ ]:
# ---------------------------------------------------------------------
#  Cascade: the slow loop (xs, ys) drives the fast loop (xf, yf), and the
#  rectified fast feedback signal returns to the slow sigmoid port with gain
#  alpha. Every cell below leaves alpha at zero, which opens the loop.
# ---------------------------------------------------------------------

Base.@kwdef mutable struct Cascade
    ks::Float64    = 3.0
    kf::Float64    = 3.0
    taus::Float64  = 60.0     # slow time constant
    tau::Float64   = 1.0      # fast time constant
    us::Float64    = 0.0      # bias of the slow loop
    u0::Float64    = 0.0      # bias of the fast loop
    beta::Float64  = 1.0      # tap gain
    alpha::Float64 = 0.0      # return gain
    tap::Symbol    = :x       # :x, :y or :w
    ys::Float64    = 1.0
    yf::Float64    = 1.0
end

function casc_rhs!(dx, x, p::Cascade, t)
    dx[1] = (-x[1] + p.ys) / p.taus
    dx[2] = (-x[2] + p.yf) / p.tau
    nothing
end

slowport(p::Cascade, x) = p.us + p.alpha * p.yf     # us + alpha y_f
tapval(p::Cascade, x)   = p.tap === :x ? x[1] :
                          p.tap === :y ? p.ys : p.ys - x[1]
fastport(p::Cascade, x) = p.u0 + p.beta * tapval(p, x)

cond_s(x, t, integ) = (p = integ.p; ue = slowport(p, x);
                       p.ys > 0 ? x[1] - (ue + 1 - 1/p.ks) : (ue - 1 + 1/p.ks) - x[1])
cond_f(x, t, integ) = (p = integ.p; ue = fastport(p, x);
                       p.yf > 0 ? x[2] - (ue + 1 - 1/p.kf) : (ue - 1 + 1/p.kf) - x[2])

# After a discrete flip, the other loop may be stranded on a branch that is no longer
# valid, because its sigmoid argument jumped instead of drifting across zero. Root finding
# cannot see that, so re-validate explicitly.
function enforce_fast!(p::Cascade, x)
    for _ in 1:2
        ue = fastport(p, x)
        if     p.yf > 0 && x[2] > ue + 1 - 1/p.kf; p.yf = -1.0
        elseif p.yf < 0 && x[2] < ue - 1 + 1/p.kf; p.yf =  1.0
        else   break
        end
    end
end

function enforce_slow!(p::Cascade, x)
    for _ in 1:2
        ue = slowport(p, x)
        if     p.ys > 0 && x[1] > ue + 1 - 1/p.ks; p.ys = -1.0
        elseif p.ys < 0 && x[1] < ue - 1 + 1/p.ks; p.ys =  1.0
        else   break
        end
    end
end

flip_s!(integ) = (integ.p.ys = -integ.p.ys; enforce_fast!(integ.p, integ.u); nothing)
flip_f!(integ) = (integ.p.yf = -integ.p.yf; enforce_slow!(integ.p, integ.u); nothing)

function simulate(p::Cascade; tspan = (0.0, 4000.0), x0 = [0.2, 0.0], dtmax = 0.05,
                  tjump = Inf, alphajump = p.alpha)
    p.ys, p.yf = 1.0, 1.0
    sv = SavedValues(Float64, Tuple{Float64,Float64})
    # alpha steps at tjump: the slow port jumps by alpha*yf instead of drifting,
    # so both branches have to be re-validated, exactly as after a discrete flip
    jump! = integ -> (integ.p.alpha = alphajump;
                      enforce_slow!(integ.p, integ.u);
                      enforce_fast!(integ.p, integ.u); nothing)
    cb = CallbackSet(ContinuousCallback(cond_s, flip_s!),
                     ContinuousCallback(cond_f, flip_f!),
                     PresetTimeCallback(isfinite(tjump) ? [tjump] : Float64[], jump!),
                     SavingCallback((x, t, integ) -> (integ.p.ys, integ.p.yf), sv))
    sol = solve(ODEProblem(casc_rhs!, x0, tspan, p), Tsit5();
                callback = cb, abstol = 1e-10, reltol = 1e-9, dtmax = dtmax)
    ys = [v[1] for v in sv.saveval]; yf = [v[2] for v in sv.saveval]
    return sol, sv.t, ys, yf
end

## Fig. 3

In [ ]:
# --- Fig. 3: the three switch positions of the open cascade -----------------
ks = kf   = 3.0
taus, tau = 60.0, 1.0
Pslow     = 2 * taus * log(2ks - 1)      # one full slow period, 193.15
t0, W     = 800.0, 2Pslow                # skip the transient, show two cycles
tjump = t0 + 218                   # return gain switched on part way through
taps  = [(:x, 1.00, 0.00, 0.1, L"\sigma_s = x_s"),
         (:w, 0.25, 0.00, 0.1, L"\sigma_s = w_s"),
         (:y, 0.45, 0.55, 0.2, L"\sigma_s = y_s")]

fmtnum(x, d) = Printf.format(Printf.Format("%.$(d)f"), iszero(x) ? 0.0 : x)

function niceticks(lo, hi; n = 4)
    raw  = (hi - lo) / n
    mag  = 10.0^floor(log10(raw))
    r    = raw / mag
    step = (r < 1.5 ? 1.0 : r < 3.0 ? 2.0 : r < 7.0 ? 5.0 : 10.0) * mag
    v    = collect(ceil(lo/step - 1e-9)*step : step : floor(hi/step + 1e-9)*step)
    d    = max(0, Int(-floor(log10(step))))
    (v, [latexstring(fmtnum(x, d)) for x in v])
end

function raster!(pl, spikes, ylo, yhi; c = myGray, lw = 2.0)
    xs, ys = Float64[], Float64[]
    for t in spikes
        append!(xs, (t, t, NaN)); append!(ys, (ylo, yhi, NaN))
    end
    plot!(pl, xs, ys; lw = lw, lc = c, label = "")
end

xt = niceticks(0, W)
panels = Plots.Plot[]
for (i, (tap, beta, u0, alpha1, lab)) in enumerate(taps)
    p = Cascade(tap = tap, beta = beta, u0 = u0, taus = taus, tau = tau, ks = ks, kf = kf)
    sol, ts, ys, yf = simulate(p; tspan = (0.0, t0 + W + 10.0), dtmax = 0.02,
                               tjump = tjump, alphajump = alpha1)

    keep  = findall(t -> t0 <= t <= t0 + W, ts)
    tw    = ts[keep] .- t0
    xsw   = [sol(t)[1] for t in ts[keep]]
    ysw   = ys[keep]
    drive = tap === :x ? u0 .+ beta .* xsw :
            tap === :y ? u0 .+ beta .* ysw :
                         u0 .+ beta .* (ysw .- xsw)
    sp = filter(t -> t0 <= t <= t0 + W, spiketimes(ts, yf)) .- t0

    lo, hi = extrema(drive)
    pad    = 0.12 * (hi - lo)
    ytop   = hi + 4pad

    pl = plot(tw, drive; lc = :black, lw = 2.0, label = "", legend = false,
              xlims = (0, W), ylims = (lo - pad, ytop + pad),
              ylabel = L"u_f",
              xlabel = i == 3 ? L"t/\tau" : "",
              xticks = i == 3 ? xt : (xt[1], fill("", length(xt[1]))),
              yticks = niceticks(lo, hi),
              tickfontsize = 12)
    hline!(pl, [-1/kf, 1/kf]; ls = :dash, lc = myRed, lw = 2.0, label = "")
    vline!(pl, [tjump-t0]; ls = :dot, lc = myBlue, lw = 2.0, label = "")
    raster!(pl, sp, hi + 2pad, ytop; lw = 3.0)
    annotate!(pl, 0.02W, ytop, text(lab, 14, :left, :bottom, :black))
    push!(panels, pl)

    i == 1 && annotate!(pl, 0.58W, ytop, text(L"\alpha = 0.2", 12, :left, :bottom, myBlue))
end

fig3 = plot(panels...; layout = (3, 1), link = :x,
            size = (600, 600), margins = 0Plots.mm,
            right_margin = 1Plots.mm, top_margin = 1Plots.mm)

mkpath("figures")
savefig(fig3, "figures/fig3.pdf")
fig3